# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/28(월) 오전 · 예외처리 · 로깅 — 실습</mark>

오늘 오전의 도착점은 **깨진 줄이 섞인 로그 파일을 넣어도 멈추지 않고 끝까지 처리하는 파서**입니다. 4교시에 `log_parser.py` 로 저장합니다.

> **파서**(parser, 파서)는 **글자로 된 기록을 프로그램이 쓸 수 있는 값으로 바꿔 주는 프로그램**입니다. 지난 시간에 CSV 한 줄을 쪼개 딕셔너리로 만든 코드가 파서입니다. 오늘은 그 파서가 깨진 줄을 만나도 멈추지 않게 만듭니다.

이 노트북은 혼자서 돌아갑니다. 지난 시간에 만든 파일이 없어도 됩니다. 아래 준비 셀이 오늘 쓸 데이터를 다시 만듭니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

### 0.1 맨 먼저 · 내 사본 만들기

1. 위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다.
2. 제목이 「사본: …」으로 바뀌면 된 것입니다.
3. 사본을 만들지 않으면 내가 쓴 코드가 저장되지 않습니다.

### 0.2 셀 실행하기

셀을 누르고 Shift + Enter를 칩니다. 왼쪽 ▶를 눌러도 같습니다. **위에서부터 차례대로** 실행합니다.

### 0.3 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 2교시 | `try`·`except` — 에러가 나도 멈추지 않는다 |
| 3교시 | 예외 이름으로 나눠 잡기 · `finally` |
| 4교시 | `logging` · `log_parser.py` 조립 |

### 0.4 막혔을 때

1. 문제 바로 위의 문법 설명 셀을 다시 봅니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다. 왼쪽 목차(☰)에서 바로 갈 수 있습니다.
3. 정답 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 먼저 스스로 해 본 뒤 엽니다.

### 0.5 오늘 만든 코드는 어디에 남나

| | 무엇 |
|---|---|
| 코랩 | 연습장 — 창을 닫으면 여기 쓴 코드는 사라집니다 |
| 내 드라이브의 `agent_core` 폴더 | 작품 보관함 — 날마다 파일이 하나씩 쌓입니다 |

1. 그래서 마지막 실습은 코드를 `.py` 파일로 저장해 드라이브에 남깁니다.
2. 셀 맨 첫 줄의 `%%writefile 이름.py` 는 「이 셀을 실행하지 말고, 이 이름의 파일로 저장하라」는 뜻입니다. 실행 결과 대신 `Writing 이름.py` 가 나옵니다.
3. `!python 이름.py` 는 저장한 파일을 실행합니다. 앞의 `!` 는 「터미널 명령」이라는 표시입니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 예외와 try·except</mark>


### <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 드라이브를 연결하고 `agent_core` 로 들어갑니다</mark>

아래 두 셀을 **맨 먼저** 실행합니다. 계정 선택 창이 뜨면 코랩을 연 계정을 고르고 허용을 누릅니다.
여기서 자리를 잡아 두면 오늘 만드는 파일이 **전부 드라이브에 바로 저장**됩니다. 나중에 옮길 일이 없습니다.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!mkdir -p /content/drive/MyDrive/agent_core
%cd /content/drive/MyDrive/agent_core
!ls


/content/drive/MyDrive/agent_core
checks.py	    hello.py		  __pycache__	   sample_logs_raw.csv
day2_csv_report.py  logs_reordered.csv	  report.json	   summary.json
day2_read_logs.py   logs_with_header.csv  sample_logs.csv  utils.py


### <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 오늘 쓸 로그 파일 두 개</mark>

아래 두 셀을 먼저 실행합니다. 코랩 안에 파일 두 개가 생깁니다.

| 파일 | 무엇 |
|---|---|
| `sample_logs.csv` | 깔끔한 로그 17건 |
| `sample_logs_broken.csv` | 같은 17건에 **깨진 줄 5건**이 섞인 것 |

칸은 `시각,계정,사건,IP` 네 개이고 머리글 줄은 없습니다.


In [3]:
%%writefile sample_logs.csv
09:01,kim01,LOGIN_OK,10.0.3.21
09:03,lee02,LOGIN_FAIL,10.0.7.5
09:05,admin,LOGIN_FAIL,10.0.9.8
09:07,park03,LOGIN_OK,10.0.4.11
09:09,kim01,LOGIN_OK,10.0.3.21
09:12,admin,LOGIN_FAIL,10.0.9.8
09:15,choi04,LOGIN_OK,10.0.6.2
09:18,lee02,LOGIN_FAIL,10.0.7.5
09:21,admin,LOGIN_FAIL,10.0.9.8
09:24,park03,LOGIN_OK,10.0.4.11
09:27,jung05,LOGIN_OK,10.0.8.30
09:30,kim01,LOGIN_OK,10.0.3.21
09:33,choi04,LOGIN_FAIL,10.0.6.2
09:36,jung05,LOGIN_OK,10.0.8.30
09:39,park03,LOGIN_OK,10.0.4.11
09:42,lee02,LOGIN_OK,10.0.7.5
09:45,kim01,LOGIN_OK,10.0.3.21


Overwriting sample_logs.csv


In [4]:
%%writefile sample_logs_broken.csv
09:01,kim01,LOGIN_OK,10.0.3.21
09:03,lee02,LOGIN_FAIL,10.0.7.5
03:22,hacker
09:05,admin,LOGIN_FAIL,10.0.9.8
09:07,park03,LOGIN_OK,10.0.4.11
09:09,kim01,LOGIN_OK,10.0.3.21

09:12,admin,LOGIN_FAIL,10.0.9.8
09:15,choi04,LOGIN_OK,10.0.6.2
서버 점검 안내: 오늘 밤 2시부터
09:18,lee02,LOGIN_FAIL,10.0.7.5
09:21,admin,LOGIN_FAIL,10.0.9.8
09:24,park03,LOGIN_OK,10.0.4.11
09:27,jung05,LOGIN_OK,10.0.8.30
??:??,unknown,LOGIN_FAIL
09:30,kim01,LOGIN_OK,10.0.3.21
09:33,choi04,LOGIN_FAIL,10.0.6.2
09:36,jung05,LOGIN_OK,10.0.8.30
09:39,park03,LOGIN_OK,10.0.4.11
09:41,guest
09:42,lee02,LOGIN_OK,10.0.7.5
09:45,kim01,LOGIN_OK,10.0.3.21


Writing sample_logs_broken.csv


In [5]:
!ls  # 파일 두 개가 보이면 준비 끝입니다


checks.py	    logs_reordered.csv	  sample_logs_broken.csv  utils.py
day2_csv_report.py  logs_with_header.csv  sample_logs.csv
day2_read_logs.py   __pycache__		  sample_logs_raw.csv
hello.py	    report.json		  summary.json


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **예외(exception)** | 실행 도중 멈추는 것과 문법이 틀린 것은 어떻게 다른가 |
| **`try`·`except`** | 무엇을 막으려고 쓰는 문법인가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 예외(exception) →
- `try`·`except` →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · try·except — 에러가 나도 멈추지 않는다</mark>


### 왜 필요한가

1. 지금까지 만든 코드는 로그가 전부 깔끔하다는 전제 위에 서 있습니다. 칸이 네 개씩 들어 있는 파일만 읽어 왔습니다.
2. 실제 로그에는 칸이 모자란 줄, 빈 줄, 로그인과 무관한 줄이 섞여 있습니다. 그런 줄을 만나면 파이썬은 그 자리에서 프로그램을 멈춥니다.
3. 로그 20만 줄 가운데 한 줄이 깨졌다고 나머지 19만 9,999줄을 못 읽으면 곤란합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 예외 | 실행 도중 생긴 문제로 프로그램이 멈추는 것 |
| `IndexError` | 리스트에 없는 인덱스를 꺼냈을 때 나는 예외 |
| `try` | 「일단 시도해 볼 코드」를 담는 블록 |
| `except` | 예외가 났을 때 대신 실행할 코드를 담는 블록 |
| 트레이스백 | 예외가 난 자리까지의 경로를 보여 주는 빨간 글 |


### 쓰는 규칙 네 가지

1. `try:` 와 `except 예외이름:` 둘 다 줄 끝에 콜론을 붙이고, 안의 코드는 네 칸 들여씁니다.
2. `try` 안에서 예외가 나면 **남은 줄을 건너뛰고** 곧바로 `except` 로 갑니다.
3. 예외가 나지 않으면 `except` 는 실행되지 않습니다.
4. `except:` 만 쓰지 않고 **예외 이름을 지정**합니다. 이름을 안 쓰면 진짜 버그까지 가려집니다.



### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 `split()` 과 인덱스</mark>

| 말 | 뜻 |
|---|---|
| `split(",")` | 문자열을 쉼표에서 나눠 **리스트**로 돌려준다 |
| 인덱스 | 리스트에서 값의 자리를 가리키는 번호. **0부터** 센다 |
| `IndexError` | 리스트에 없는 인덱스를 꺼냈을 때 나는 예외 |

```python
text = "seoul,busan,daegu"
cities = text.split(",")
print(cities)        # ['seoul', 'busan', 'daegu']
print(cities[0])     # seoul
print(cities[2])     # daegu
```

값이 3개면 쓸 수 있는 인덱스는 **0·1·2** 까지입니다. `cities[3]` 은 `IndexError` 입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts)
```

막히면 바로 위 `2.1 split() 과 인덱스` 설명을 다시 봅니다.


In [6]:
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts)


['09:01', 'kim01', 'LOGIN_OK']


✅ `['09:01', 'kim01', 'LOGIN_OK']`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 `parts[3]` 입니다. 값은 앞 문제와 똑같습니다.

```python
line = "09:01,kim01,LOGIN_OK"
parts = line.split(",")
print(parts[3])
```

> ⚠ 이 코드는 **예외가 나서 멈춥니다.** 실행하지 않고 결과만 적은 뒤 아래 ✅ 로 맞춰 봅니다.

막히면 바로 위 `2.1 split() 과 인덱스` 설명을 다시 봅니다.


✅ `IndexError: list index out of range`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 줄을 리스트로 나누기</font></h3></td></tr></table>

주어진 줄을 쉼표로 나눠 **리스트로 만들어 출력**하시오.

1. `line` 을 쉼표에서 나눕니다.
2. 나눈 결과를 이름 하나에 담습니다.
3. 그 이름을 `print()` 로 출력합니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:05,admin,LOGIN_FAIL,10.0.9.8"` |
| 🎯 나와야 하는 결과 | `['09:05', 'admin', 'LOGIN_FAIL', '10.0.9.8']` |

**💡 힌트**

1. 문자열에 `.split(",")` 를 붙이면 리스트가 나옵니다.
2. 나온 리스트를 이름 하나에 담아 둡니다.
3. 그 이름을 `print()` 안에 넣습니다.


In [7]:
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts)

['09:05', 'admin', 'LOGIN_FAIL', '10.0.9.8']


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 계정 이름만 꺼내기</font></h3></td></tr></table>

같은 줄에서 **계정 이름**만 출력하시오.

1. 줄을 쉼표로 나눕니다.
2. 계정 이름은 앞에서 **두 번째** 칸입니다. 인덱스는 0부터 센다는 것을 기억합니다.
3. 그 칸 하나만 출력합니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:05,admin,LOGIN_FAIL,10.0.9.8"` |
| 🎯 나와야 하는 결과 | `admin` |

**💡 힌트**

1. 먼저 문제 2-3처럼 리스트로 만듭니다.
2. 리스트에서 값 하나를 꺼낼 때는 대괄호에 번호를 씁니다.
3. 번호는 0부터 셉니다. 두 번째 칸의 번호는 1입니다.


In [8]:
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts[1])

admin


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 시각과 IP 를 한 줄에</font></h3></td></tr></table>

같은 줄에서 **시각과 IP** 를 한 줄에 출력하시오.

1. 줄을 쉼표로 나눕니다.
2. 시각은 첫 번째 칸, IP 는 네 번째 칸입니다.
3. `print(a, b)` 처럼 쉼표로 나열하면 한 줄에 둘이 나옵니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:05,admin,LOGIN_FAIL,10.0.9.8"` |
| 🎯 나와야 하는 결과 | `09:05 10.0.9.8` |

**💡 힌트**

1. 리스트로 만드는 것까지는 앞 문제와 같습니다.
2. 첫 번째 칸의 번호는 0, 네 번째 칸의 번호는 3입니다.
3. `print(a, b)` 처럼 쉼표로 나열하면 한 줄에 둘이 나옵니다.


In [9]:
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts[0], parts[3])

09:05 10.0.9.8


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰어도 됩니다. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-1 · 한 문장으로 만들기</font></h3></td></tr></table>

로그 한 줄에서 계정과 IP 를 꺼내 **한 문장으로** 출력하시오.

1. 줄을 쉼표로 나눕니다.
2. 계정과 IP 칸을 꺼냅니다.
3. f-string 으로 한 문장을 만들어 출력합니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:05,admin,LOGIN_FAIL,10.0.9.8"` |
| 🎯 나와야 하는 결과 | `admin 의 접속 IP 는 10.0.9.8 입니다` |

**💡 힌트**

1. f-string 은 따옴표 앞에 `f` 를 붙입니다.
2. 중괄호 안에 꺼낸 값을 넣습니다.
3. 중괄호 밖의 글자는 그대로 나옵니다.


In [10]:
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-2 · 시와 분을 따로 꺼내기</font></h3></td></tr></table>

같은 줄에서 시각을 꺼낸 뒤, **시와 분을 따로** 출력하시오.

1. 줄을 쉼표로 나눠 시각 칸을 꺼냅니다.
2. 그 시각을 다시 **콜론**으로 나눕니다.
3. 앞 칸과 뒤 칸을 각각 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `09` · `05` 두 줄 |

**💡 힌트**

1. 시각 칸을 먼저 꺼냅니다.
2. 꺼낸 시각에 다시 `.split(":")` 을 붙입니다.
3. 나온 리스트의 0번과 1번을 각각 출력합니다.


In [ ]:
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 `try` 와 `except`</mark>

```python
try:
    시도할 코드
except 예외이름:
    예외가 났을 때 실행할 코드
```

예외가 나면 `try` 의 남은 줄을 건너뛰고 `except` 로 갑니다. 프로그램은 멈추지 않고 계속 돌아갑니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-6 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
try:
    n = int("abc")
    print(n)
except ValueError:
    print("숫자가 아닙니다")

print("프로그램은 계속 돌아갑니다")
```

막히면 바로 위 `2.2 try 와 except` 설명을 다시 봅니다.


In [11]:
try:
    n = int("abc")
    print(n)
except ValueError:
    print("숫자가 아닙니다")

print("프로그램은 계속 돌아갑니다")


숫자가 아닙니다
프로그램은 계속 돌아갑니다


✅ `숫자가 아닙니다 · 프로그램은 계속 돌아갑니다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-7 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 목록을 반복합니다. 가운데 줄이 깨져 있습니다.

```python
lines = ["09:01,kim01,LOGIN_OK", "03:22,hacker", "09:05,lee02,LOGIN_FAIL"]

for line in lines:
    parts = line.split(",")
    try:
        print(parts[1], parts[2])
    except IndexError:
        print("깨진 줄 건너뜀:", line)

print("끝까지 읽었습니다")
```

막히면 바로 위 `2.2 try 와 except` 설명을 다시 봅니다.


In [12]:
lines = ["09:01,kim01,LOGIN_OK", "03:22,hacker", "09:05,lee02,LOGIN_FAIL"]

for line in lines:
    parts = line.split(",")
    try:
        print(parts[1], parts[2])
    except IndexError:
        print("깨진 줄 건너뜀:", line)

print("끝까지 읽었습니다")


kim01 LOGIN_OK
깨진 줄 건너뜀: 03:22,hacker
lee02 LOGIN_FAIL
끝까지 읽었습니다


✅ `kim01 LOGIN_OK · 깨진 줄 건너뜀: 03:22,hacker · lee02 LOGIN_FAIL · 끝까지 읽었습니다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-8 · 없는 칸을 꺼낼 때</font></h3></td></tr></table>

주어진 값에서 **네 번째 칸**을 출력하시오. 칸이 없으면 대신 `칸이 모자랍니다` 를 출력합니다.

1. `line` 을 쉼표로 나눠 리스트로 만듭니다.
2. 네 번째 칸을 출력하는 줄을 `try:` 안에 둡니다.
3. `except IndexError:` 안에서 `칸이 모자랍니다` 를 출력합니다.
4. 맨 마지막에 `확인 끝` 을 출력합니다. 이 줄은 들여쓰지 않습니다.

| | |
|---|---|
| 주어지는 값 | `line = "09:41,guest"` |
| 🎯 나와야 하는 결과 | `칸이 모자랍니다` · `확인 끝` |

**💡 힌트**

1. `try:` 줄 끝에 콜론을 붙이고 아래는 네 칸 들여씁니다.
2. `except` 뒤에는 예외 이름을 적습니다. 없는 번호를 꺼낼 때 나는 이름입니다.
3. 마지막 줄은 들여쓰지 않아야 `try` 바깥이 됩니다.


In [13]:
line = "09:41,guest"
parts = line.split(",")

try:
    print(parts[3])
except IndexError:
    print("칸이 모자랍니다")

print("확인 끝")

칸이 모자랍니다
확인 끝


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-9 · 깨진 파일을 끝까지 읽기</font></h3></td></tr></table>

`sample_logs_broken.csv` 를 한 줄씩 읽어 **계정 이름과 IP** 를 출력하시오. 깨진 줄에서 멈추면 안 됩니다.

1. `with open("sample_logs_broken.csv", encoding="utf-8") as f:` 로 파일을 엽니다.
2. `for line in f:` 로 한 줄씩 반복합니다.
3. 반복 안에서 `line.strip().split(",")` 으로 줄을 나눕니다.
4. 계정 이름과 IP 를 출력하는 줄을 `try:` 안에 둡니다. IP 는 마지막 칸입니다.
5. `except IndexError:` 에서는 아무것도 출력하지 않고 `pass` 만 씁니다.

| | |
|---|---|
| 주어지는 값 | 준비 셀에서 만든 `sample_logs_broken.csv` |
| 🎯 나와야 하는 결과 | 정상 17건의 계정·IP 가 출력되고 **끝까지 실행됨** |

**💡 힌트**

1. 파일은 `with open(...) as f:` 로 엽니다.
2. `for line in f:` 가 한 줄씩 돌려줍니다.
3. 출력하는 줄만 `try` 안에 두면 됩니다. 나누는 줄은 밖에 둬도 됩니다.


In [14]:
# 여기에 코드를 입력하세요
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            print(parts[1], parts[3])
        except IndexError:
            pass

kim01 10.0.3.21
lee02 10.0.7.5
admin 10.0.9.8
park03 10.0.4.11
kim01 10.0.3.21
admin 10.0.9.8
choi04 10.0.6.2
lee02 10.0.7.5
admin 10.0.9.8
park03 10.0.4.11
jung05 10.0.8.30
kim01 10.0.3.21
choi04 10.0.6.2
jung05 10.0.8.30
park03 10.0.4.11
lee02 10.0.7.5
kim01 10.0.3.21


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-10 · 깨진 줄이 몇 건인가</font></h3></td></tr></table>

같은 파일을 다시 읽으면서, **깨진 줄의 개수**를 세어 마지막에 한 번 출력하시오.

1. 반복을 시작하기 전에 숫자를 담을 이름을 하나 만들고 `0` 을 넣습니다.
2. `try` 안에서는 아무것도 출력하지 않고 `parts[3]` 을 꺼내기만 합니다.
3. `except IndexError:` 안에서 그 숫자를 1 늘립니다.
4. 반복이 끝난 뒤 그 숫자를 출력합니다.

| | |
|---|---|
| 주어지는 값 | `sample_logs_broken.csv` |
| 🎯 나와야 하는 결과 | `깨진 줄 5건` |

**💡 힌트**

1. 숫자를 담을 이름을 반복 **전에** 만듭니다.
2. `except` 안에서 그 이름에 1을 더해 다시 담습니다.
3. 출력은 반복이 끝난 뒤 한 번만 합니다.


In [16]:
# 여기에 코드를 입력하세요
broken_count = 0

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            parts[3]
        except IndexError:
            broken_count = broken_count + 1

print(f"깨진 줄 {broken_count}건")

깨진 줄 5건


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰고 3교시로 가세요. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-3 · 깨진 줄을 따로 모으기</font></h3></td></tr></table>

깨진 줄의 **개수**가 아니라 **내용**을 리스트에 모아 마지막에 출력하시오.

1. 반복을 시작하기 전에 빈 리스트를 하나 만듭니다.
2. `except IndexError:` 안에서 그 줄을 리스트에 `append` 합니다. 줄 끝의 줄바꿈은 `strip()` 으로 없앱니다.
3. 반복이 끝난 뒤 리스트를 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 깨진 줄 5개가 담긴 리스트 |

**💡 힌트**

1. 빈 리스트는 대괄호만 써서 만듭니다.
2. 리스트에 값을 더할 때는 `.append()` 를 씁니다.
3. 줄 끝의 줄바꿈은 `.strip()` 으로 없앱니다.


In [17]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-4 · 정상 줄만 딕셔너리로</font></h3></td></tr></table>

깨진 줄은 건너뛰고, **정상 줄만** 딕셔너리로 바꿔 리스트에 모으시오.

1. 칸 이름은 `time`·`user`·`event`·`ip` 네 개로 합니다.
2. 딕셔너리를 만드는 줄을 `try` 안에 둡니다.
3. 반복이 끝난 뒤 모은 개수를 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정상 로그 17건` |

**💡 힌트**

1. 딕셔너리는 중괄호로 만들고 칸 이름과 값을 콜론으로 잇습니다.
2. 딕셔너리를 만드는 줄 자체를 `try` 안에 둬야 합니다.
3. 개수는 `len()` 으로 셉니다.


In [18]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-5 · 함수로 묶기</font></h3></td></tr></table>

줄 하나를 딕셔너리로 바꾸는 함수를 만들고, 그 함수를 `try` 안에서 부르시오.

1. 함수 이름은 `parse_line`, 매개변수는 `line` 하나로 합니다.
2. 함수 안에서 줄을 나누고 네 칸짜리 딕셔너리를 `return` 합니다.
3. 반복 안에서 `try` 로 감싸 부르고, 깨진 줄은 건너뜁니다.
4. 예외가 함수 **안**에서 나도 `except` 가 잡는다는 것을 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정상 로그 17건` |

**💡 힌트**

1. 함수는 `def 이름(매개변수):` 로 시작합니다.
2. 돌려줄 값은 `return` 뒤에 적습니다.
3. 예외가 함수 안에서 나도 부른 쪽의 `except` 가 잡습니다.


In [19]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `try:` | 일단 시도할 코드를 담는다 |
| `except IndexError:` | `IndexError` 가 났을 때 대신 실행한다 |
| `pass` | 아무것도 하지 않고 넘어간다 |
| 예외가 없으면 | `except` 는 실행되지 않는다 |
| 예외가 나면 | `try` 의 남은 줄을 건너뛰고 `except` 로 간다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 예외 이름 집어 잡기 · finally</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **`ValueError`** | 어떤 상황에서 나는 예외인가 |
| **`finally`** | `try`·`except` 에서 어떤 자리를 맡는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- `ValueError` →
- `finally` →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · 예외를 이름별로 나눠 잡는다</mark>


### 왜 필요한가

1. 2교시에는 깨진 줄을 한 가지로만 봤습니다. `IndexError` 하나로 전부 묶어 버렸습니다.
2. 실제 로그가 깨지는 방식은 여럿입니다. 칸이 모자란 줄, 시각이 `??:??` 인 줄, 로그와 무관한 줄이 섞입니다.
3. 무엇이 왜 깨졌는지 나눠서 알아야 고칠 곳을 찾을 수 있습니다. 그래서 예외를 이름별로 따로 잡습니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| `ValueError` | 갈래는 맞지만 모양이 맞지 않는 값을 넘겼을 때 나는 예외 |
| `FileNotFoundError` | 없는 파일을 열려고 했을 때 나는 예외 |
| `finally` | 예외가 나든 나지 않든 **반드시** 실행되는 블록 |


### 쓰는 규칙 네 가지

1. `except` 는 여러 개를 이어 쓸 수 있습니다. **위에서부터** 맞는 이름 하나만 실행됩니다.
2. 예외 이름이 맞지 않으면 그 `except` 는 잡지 못하고 프로그램이 멈춥니다.
3. `finally` 는 맨 아래에 한 번만 씁니다. `try` 가 성공해도, 예외가 나도 실행됩니다.
4. `except:` 처럼 이름 없이 쓰지 않습니다. 진짜 버그까지 가려집니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 `int()` 와 `ValueError`</mark>

| 말 | 뜻 |
|---|---|
| `int("9")` | 글자를 정수로 바꾼다 |
| `ValueError` | 숫자로 못 바꾸는 글자를 넘겼을 때 나는 예외 |
| `"09:01".split(":")` | 콜론에서 나눠 `['09', '01']` 을 돌려준다 |

```python
temp = "21:30"
hour = int(temp.split(":")[0])
print(hour)          # 21
```

`int("abc")` 나 `int("")` 는 `ValueError` 입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

값만 바뀌었습니다. 시각이 깨진 줄입니다.

```python
time = "??:??"
hour = int(time.split(":")[0])
print(hour)
```

> ⚠ 이 코드는 **예외가 나서 멈춥니다.** 실행하지 않고 결과만 적은 뒤 아래 ✅ 로 맞춰 봅니다.

막히면 바로 위 `3.1 int() 와 ValueError` 설명을 다시 봅니다.


✅ `ValueError: invalid literal for int() with base 10: '??'`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 무엇이 보일까요</font></h3></td></tr></table>

같은 코드를 `try` 로 감쌌습니다.

```python
time = "??:??"

try:
    hour = int(time.split(":")[0])
    print(hour)
except ValueError:
    print("시각을 읽을 수 없습니다")
```

막히면 바로 위 `3.1 int() 와 ValueError` 설명을 다시 봅니다.


In [20]:
time = "??:??"

try:
    hour = int(time.split(":")[0])
    print(hour)
except ValueError:
    print("시각을 읽을 수 없습니다")


시각을 읽을 수 없습니다


✅ `시각을 읽을 수 없습니다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 시각에서 시만 꺼내기</font></h3></td></tr></table>

주어진 시각에서 **시**만 숫자로 꺼내 출력하시오.

1. `time` 을 콜론에서 나눕니다.
2. 앞쪽 칸이 시입니다.
3. `int()` 로 숫자로 바꿔 출력합니다.

| | |
|---|---|
| 주어지는 값 | `time = "21:30"` |
| 🎯 나와야 하는 결과 | `21` |

**💡 힌트**

1. 콜론으로 나누면 리스트가 됩니다.
2. 앞쪽 칸의 번호는 0입니다.
3. 글자를 숫자로 바꿀 때는 `int()` 를 씁니다.


In [21]:
time = "21:30"

print(int(time.split(":")[0]))



21


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 로그 한 줄에서 시 꺼내기</font></h3></td></tr></table>

로그 한 줄에서 **시**를 숫자로 꺼내 출력하시오.

1. 줄을 쉼표로 나눠 시각 칸을 꺼냅니다.
2. 그 시각을 다시 콜론에서 나눕니다.
3. 앞쪽 칸을 `int()` 로 바꿔 출력합니다.

| | |
|---|---|
| 주어지는 값 | `line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"` |
| 🎯 나와야 하는 결과 | `3` |

**💡 힌트**

1. 먼저 쉼표로 나눠 시각 칸을 꺼냅니다.
2. 꺼낸 시각을 다시 콜론으로 나눕니다.
3. 두 번 나눈 결과에 `int()` 를 씌웁니다.


In [23]:
line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"
parts = line.split(",")

print(int(parts[0].split(":")[0]))

3


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · 야간인지 판단하기</font></h3></td></tr></table>

같은 줄에서 시를 꺼낸 뒤, **0시부터 6시 사이**이면 `야간` 을 출력하시오.

1. 문제 3-4와 같은 방법으로 시를 꺼냅니다.
2. `if` 로 0 이상이고 6 이하인지 확인합니다. 두 조건은 `and` 로 잇습니다.
3. 맞으면 `야간` 을 출력합니다.

| | |
|---|---|
| 주어지는 값 | `line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"` |
| 🎯 나와야 하는 결과 | `야간` |

**💡 힌트**

1. 시를 꺼내는 것까지는 앞 문제와 같습니다.
2. `if` 줄 끝에 콜론을 붙입니다.
3. 두 조건은 `and` 로 잇습니다.


In [24]:
line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"
parts = line.split(",")
hour = int(parts[0].split(":")[0])

if hour >= 0 and hour <= 6:
    print("야간")


야간


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰어도 됩니다. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-1 · 시각 목록에서 시만 모으기</font></h3></td></tr></table>

시각이 담긴 목록을 하나씩 읽어, **시**만 숫자로 출력하시오.

1. `for` 로 `times` 를 하나씩 반복합니다.
2. 반복 안에서 콜론으로 나눠 앞쪽 칸을 `int()` 로 바꿉니다.
3. 바꾼 값을 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `9` · `3` · `21` 세 줄 |

**💡 힌트**

1. `for` 로 목록을 하나씩 돌립니다.
2. 반복 안에서 할 일은 네 칸 들여씁니다.
3. 출력을 반복 안에 둬야 세 줄이 나옵니다.


In [ ]:
times = ["09:01", "03:22", "21:30"]


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-2 · 야간인지 주간인지</font></h3></td></tr></table>

같은 목록을 읽어, 0시부터 6시 사이면 `야간`, 아니면 `주간` 을 출력하시오.

1. 도전 ⭐3-1과 같은 방법으로 시를 꺼냅니다.
2. `if` 와 `else` 로 두 갈래를 나눕니다.
3. 시와 판단을 한 줄에 함께 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `9 주간` · `3 야간` · `21 주간` |

**💡 힌트**

1. `if` 아래에 `else:` 를 같은 들여쓰기로 둡니다.
2. `print(a, b)` 로 두 값을 한 줄에 냅니다.
3. 조건에 맞지 않는 값이 `else` 로 갑니다.


In [ ]:
times = ["09:01", "03:22", "21:30"]


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 `except` 를 여러 개 쓰는 법</mark>

```python
try:
    시도할 코드
except ValueError:
    값의 모양이 틀렸을 때
except IndexError:
    없는 인덱스를 꺼냈을 때
finally:
    예외가 나든 나지 않든 언제나
```

위에서부터 맞는 `except` **하나만** 실행됩니다. `finally` 는 언제나 실행됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-6 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 세 줄을 차례대로 따라갑니다.

```python
for line in ["09:01,kim01,LOGIN_OK,10.0.3.21", "??:??,unknown,LOGIN_FAIL", "09:41,guest"]:
    parts = line.split(",")
    try:
        hour = int(parts[0].split(":")[0])
        print(hour, parts[3])
    except ValueError:
        print("시각이 깨진 줄:", line)
    except IndexError:
        print("칸이 모자란 줄:", line)
```

막히면 바로 위 `3.2 except 를 여러 개 쓰는 법` 설명을 다시 봅니다.


In [25]:
for line in ["09:01,kim01,LOGIN_OK,10.0.3.21", "??:??,unknown,LOGIN_FAIL", "09:41,guest"]:
    parts = line.split(",")
    try:
        hour = int(parts[0].split(":")[0])
        print(hour, parts[3])
    except ValueError:
        print("시각이 깨진 줄:", line)
    except IndexError:
        print("칸이 모자란 줄:", line)


9 10.0.3.21
시각이 깨진 줄: ??:??,unknown,LOGIN_FAIL
칸이 모자란 줄: 09:41,guest


✅ `9 10.0.3.21 · 시각이 깨진 줄: ??:??,unknown,LOGIN_FAIL · 칸이 모자란 줄: 09:41,guest`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-7 · 무엇이 보일까요</font></h3></td></tr></table>

`finally` 가 붙었습니다. 두 값을 차례로 넣습니다.

```python
for time in ["09:01", "??:??"]:
    try:
        print(int(time.split(":")[0]))
    except ValueError:
        print("읽을 수 없음")
    finally:
        print("--- 한 줄 확인 끝")
```

막히면 바로 위 `3.2 except 를 여러 개 쓰는 법` 설명을 다시 봅니다.


In [26]:
for time in ["09:01", "??:??"]:
    try:
        print(int(time.split(":")[0]))
    except ValueError:
        print("읽을 수 없음")
    finally:
        print("--- 한 줄 확인 끝")


9
--- 한 줄 확인 끝
읽을 수 없음
--- 한 줄 확인 끝


✅ `9 · --- 한 줄 확인 끝 · 읽을 수 없음 · --- 한 줄 확인 끝`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-8 · 야간 로그만 골라내기</font></h3></td></tr></table>

`sample_logs_broken.csv` 를 읽어 **시각이 0시부터 6시 사이**인 줄의 계정을 출력하시오.

1. 파일을 열고 `for line in f:` 로 한 줄씩 반복합니다.
2. 반복 안에서 줄을 쉼표로 나눕니다.
3. `try` 안에서 시각의 **시**를 숫자로 바꾸고, 0 이상 6 이하이면 계정을 출력합니다.
4. `except ValueError:` 에서는 `pass` 만 씁니다.
5. `except IndexError:` 에서도 `pass` 만 씁니다.

| | |
|---|---|
| 주어지는 값 | `sample_logs_broken.csv` |
| 🎯 나와야 하는 결과 | `hacker` |

**💡 힌트**

1. 파일을 열고 한 줄씩 도는 뼈대는 문제 2-9와 같습니다.
2. 시를 꺼내는 방법은 문제 3-4와 같습니다.
3. `except` 를 두 개 이어 쓰고 안에는 `pass` 만 둡니다.


In [27]:
# 여기에 코드를 입력하세요
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            hour = int(parts[0].split(":")[0])
            if hour >= 0 and hour <= 6:
                print(parts[1])
        except ValueError:
            pass
        except IndexError:
            pass


hacker


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-9 · 깨진 이유를 나눠 세기</font></h3></td></tr></table>

같은 파일을 읽으면서 깨진 줄을 **이유별로** 세어 마지막에 두 줄로 출력하시오.

1. 숫자를 담을 이름 두 개를 만들고 각각 `0` 을 넣습니다.
2. `try` 안에서 시각의 시를 숫자로 바꾸고, 이어서 `parts[3]` 을 꺼냅니다.
3. `except ValueError:` 에서 첫 번째 숫자를 1 늘립니다.
4. `except IndexError:` 에서 두 번째 숫자를 1 늘립니다.
5. 반복이 끝난 뒤 두 숫자를 각각 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `시각이 깨진 줄 3건` · `칸이 모자란 줄 2건` |

**💡 힌트**

1. 숫자를 담을 이름 두 개를 반복 전에 만듭니다.
2. `try` 안에서는 출력하지 않고 값만 꺼내 봅니다.
3. 어떤 `except` 로 가는지에 따라 다른 이름을 늘립니다.


In [28]:
# 여기에 코드를 입력하세요
bad_time = 0
bad_column = 0

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            int(parts[0].split(":")[0])
            parts[3]
        except ValueError:
            bad_time = bad_time + 1
        except IndexError:
            bad_column = bad_column + 1

print(f"시각이 깨진 줄 {bad_time}건")
print(f"칸이 모자란 줄 {bad_column}건")


시각이 깨진 줄 3건
칸이 모자란 줄 2건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-10 · 없는 파일을 열었을 때</font></h3></td></tr></table>

있지도 않은 `sample_logs_2025.csv` 를 열어 보고, 없으면 안내를 출력하시오. 마지막 줄은 어느 경우에도 나와야 합니다.

1. `try` 안에서 파일을 열고 첫 줄을 읽어 출력합니다.
2. `except FileNotFoundError:` 에서 `파일이 없습니다` 를 출력합니다.
3. `finally:` 에서 `확인 끝` 을 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `파일이 없습니다` · `확인 끝` |

**💡 힌트**

1. 파일을 여는 `with` 줄 전체를 `try` 안에 둡니다.
2. `except` 뒤 이름은 없는 파일을 열 때 나는 이름입니다.
3. `finally:` 는 `except` 아래에 같은 들여쓰기로 둡니다.


In [29]:
# 여기에 코드를 입력하세요
try:
    with open("sample_logs_2025.csv", encoding="utf-8") as f:
        print(f.readline())
except FileNotFoundError:
    print("파일이 없습니다")
finally:
    print("확인 끝")


파일이 없습니다
확인 끝


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰고 4교시로 가세요. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-3 · 깨진 줄을 이유별로 모으기</font></h3></td></tr></table>

개수가 아니라 **줄 내용**을 두 리스트에 나눠 담고, 각각 출력하시오.

1. 빈 리스트 두 개를 만듭니다.
2. 문제 3-9와 같은 방식으로 나누되, 숫자를 늘리는 대신 줄을 `append` 합니다.
3. 반복이 끝난 뒤 두 리스트를 각각 출력합니다.

**💡 힌트**

1. 빈 리스트 두 개를 반복 전에 만듭니다.
2. 문제 3-9와 뼈대가 같습니다. 숫자 대신 줄을 담습니다.
3. `.append()` 에 넣기 전에 `.strip()` 을 씁니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-4 · 함수 안에서 두 예외 잡기</font></h3></td></tr></table>

줄 하나를 딕셔너리로 바꾸는 `parse_line` 함수를 만들고, 부르는 쪽에서 두 예외를 나눠 잡으시오.

1. 함수 안에서 줄을 나누고, 시각의 시를 숫자로 바꿔 `hour` 칸에 함께 담습니다.
2. 칸 이름은 `time`·`user`·`event`·`ip`·`hour` 다섯 개입니다.
3. 부르는 쪽에서 `except ValueError:` 와 `except IndexError:` 를 각각 두고, 정상 줄만 리스트에 모읍니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `정상 로그 17건` |

**💡 힌트**

1. 함수 안에서 `int()` 를 쓰면 거기서 `ValueError` 가 납니다.
2. 딕셔너리에 칸을 하나 더 넣습니다.
3. 부르는 쪽에 `except` 를 두 개 이어 씁니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-5 · 몇 줄을 시도했나</font></h3></td></tr></table>

`finally` 로 **시도한 줄 수**를 세시오. 성공한 줄과 깨진 줄을 모두 합한 수입니다.

1. 숫자를 담을 이름을 하나 만들고 `0` 을 넣습니다.
2. `finally:` 안에서 그 숫자를 1 늘립니다.
3. 반복이 끝난 뒤 시도한 줄 수와 정상 줄 수를 각각 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `시도한 줄 22건` · `정상 로그 17건` |

**💡 힌트**

1. `finally:` 는 성공해도 실패해도 실행됩니다.
2. 숫자를 늘리는 줄을 `finally` 안에 둡니다.
3. 정상 줄 수는 리스트의 `len()` 으로 셉니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `except ValueError:` | 값의 모양이 틀렸을 때 |
| `except IndexError:` | 없는 인덱스를 꺼냈을 때 |
| `except FileNotFoundError:` | 없는 파일을 열려 했을 때 |
| `except` 여러 개 | 위에서부터 맞는 것 **하나만** 실행 |
| `finally:` | 예외가 나든 나지 않든 언제나 실행 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · logging · log_parser.py 조립</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 코랩에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **로깅(logging)** | `print` 로 찍는 것과 무엇이 다른가 |
| **로그 레벨** | `INFO`·`WARNING`·`ERROR` 를 왜 나누어 두나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 로깅(logging) →
- 로그 레벨 →

맨 처음에 「드라이브에 사본 저장」을 눌렀다면 여기 적은 것은 내 드라이브의 노트북에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · logging — 기록을 파일에 남긴다</mark>


### 왜 필요한가

1. 지금까지 깨진 줄은 `print` 로 화면에 알렸습니다. 화면 출력은 그 순간 보일 뿐 어디에도 저장되지 않습니다.
2. 관제 프로그램은 새벽에 혼자 돌아갑니다. 아침에 출근해서 「어젯밤 깨진 줄이 몇 건이었지」를 확인할 방법이 없습니다.
3. 기록에는 **언제**와 **얼마나 심각한지**가 함께 남아야 합니다. 그것을 대신해 주는 도구가 `logging` 입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 파서 | 글자로 된 기록을 프로그램이 쓸 수 있는 값으로 바꾸는 프로그램 |
| `logging` | 프로그램의 활동 기록을 남기는 파이썬 기본 도구 |
| `basicConfig` | 어디에·무엇부터·어떤 모양으로 남길지 정하는 설정 |
| 심각도 | 기록의 등급. `INFO` → `WARNING` → `ERROR` 순으로 심각하다 |
| `level` | 어느 심각도부터 남길지 정하는 설정값 |


### 쓰는 규칙 네 가지

1. 설정은 파일 맨 위에서 **한 번만** 합니다.
2. `filename` 을 적어야 화면이 아니라 파일에 남습니다.
3. `level=logging.INFO` 를 빼면 기본값이 `WARNING` 이라 `logging.info` 가 하나도 안 남습니다.
4. 오늘은 `logging` 을 **`.py` 파일로 저장해 실행**합니다. 코랩 셀에서 바로 부르면 설정이 무시될 수 있습니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 `logging` 설정과 세 가지 심각도</mark>

```python
import logging

logging.basicConfig(
    filename="agent.log",                              # 기록을 남길 파일
    level=logging.INFO,                                # INFO 부터 남긴다
    format="%(asctime)s %(levelname)s %(message)s",    # 시각 심각도 내용
    encoding="utf-8",
)

logging.info("파서 시작")         # 정상 동작 기록
logging.warning("깨진 줄 건너뜀")   # 주의할 일
logging.error("파일을 열 수 없음")  # 심각한 오류
```

설정 네 줄은 그대로 복사해서 씁니다. `!cat 파일이름` 은 파일 내용을 화면에 보여 주는 터미널 명령입니다.


아래 셀을 먼저 실행해 `log_demo.py` 를 만듭니다. `Writing log_demo.py` 가 나오면 된 것입니다.


In [30]:
%%writefile log_demo.py
import logging

logging.basicConfig(
    filename="agent.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("기록을 시작합니다")
logging.warning("주의할 일이 있었습니다")
print("화면에는 이 줄만 보입니다")


Writing log_demo.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

위에서 만든 `log_demo.py` 를 실행합니다. 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
!python log_demo.py
```

막히면 바로 위 `4.1 logging 설정과 세 가지 심각도` 설명을 다시 봅니다.


In [31]:
!python log_demo.py


화면에는 이 줄만 보입니다


✅ `화면에는 이 줄만 보입니다`


기록 두 줄은 화면에 없습니다. `agent.log` 파일로 갔습니다.


아래 두 셀을 먼저 실행해 `nolevel.log` 를 만듭니다. **`level` 을 일부러 빼고** 설정했습니다.


In [32]:
%%writefile nolevel_demo.py
import logging

logging.basicConfig(
    filename="nolevel.log",
    format="%(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("info 로 남긴 줄")
logging.warning("이 줄은 남을까요")


Writing nolevel_demo.py


In [33]:
!python nolevel_demo.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 무엇이 보일까요</font></h3></td></tr></table>

`level` 을 뺀 채 실행한 결과입니다. `nolevel.log` 에 무엇이 남았을지 적어 보세요.

```python
!cat nolevel.log
```

막히면 바로 위 `4.1 logging 설정과 세 가지 심각도` 설명을 다시 봅니다.


In [34]:
!cat nolevel.log


WARNING 이 줄은 남을까요


✅ `WARNING 이 줄은 남을까요 — 한 줄만`


기본 심각도가 `WARNING` 이라 `info` 는 남지 않았습니다. 「info 가 안 찍혀요」의 가장 흔한 원인입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · 기록 한 줄 남기기</font></h3></td></tr></table>

`logging` 으로 경고 한 줄을 파일에 남기는 `.py` 파일을 만드시오.

1. 새 코드 셀 맨 첫 줄에 `%%writefile my_log.py` 를 씁니다.
2. `import logging` 을 쓰고, 설정 네 줄을 그대로 씁니다. 기록 파일 이름은 `my.log` 로 합니다.
3. `logging.warning` 으로 `첫 기록` 을 남깁니다.
4. 셀을 실행해 `Writing my_log.py` 를 확인합니다.
5. 새 셀에서 `!python my_log.py` 와 `!cat my.log` 를 차례로 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `my.log` 에 `WARNING 첫 기록` 한 줄 |

**💡 힌트**

1. 셀 맨 첫 줄에 `%%writefile` 을 쓰면 그 셀은 실행되지 않고 파일로 저장됩니다.
2. 설정은 `logging.basicConfig(` 로 시작해 `)` 로 닫습니다.
3. `filename` 을 적어야 화면이 아니라 파일에 남습니다.


In [37]:
%%writefile my_log.py
import logging

logging.basicConfig(
    filename="my.log",
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.warning("첫 기록")


Overwriting my_log.py


In [41]:
!python my_log.py
!cat my.log

2026-09-28 14:26:10,506 INFO 시작
2026-09-28 14:26:10,508 WARNING 주의
2026-09-28 14:26:10,508 ERROR 오류


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · 심각도 세 가지 남기기</font></h3></td></tr></table>

같은 파일에 심각도 세 가지를 모두 남기시오.

1. `my_log.py` 를 다시 만듭니다. 설정에 `level=logging.INFO` 를 넣습니다.
2. `logging.info` 로 `시작`, `logging.warning` 으로 `주의`, `logging.error` 로 `오류` 를 차례로 남깁니다.
3. `!python my_log.py` 와 `!cat my.log` 로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `my.log` 에 INFO · WARNING · ERROR 세 줄이 이어 붙는다 |

**💡 힌트**

1. 설정에 `level=logging.INFO` 를 넣어야 `info` 가 남습니다.
2. 세 줄은 `logging.` 뒤 이름만 다릅니다.
3. 파일을 고쳤으면 `%%writefile` 셀을 다시 실행해야 합니다.


In [40]:
%%writefile my_log.py
import logging

logging.basicConfig(
    filename="my.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("시작")
logging.warning("주의")
logging.error("오류")

Overwriting my_log.py


In [42]:
!python my_log.py
!cat my.log

2026-09-28 14:26:10,506 INFO 시작
2026-09-28 14:26:10,508 WARNING 주의
2026-09-28 14:26:10,508 ERROR 오류
2026-09-28 14:26:28,695 INFO 시작
2026-09-28 14:26:28,696 WARNING 주의
2026-09-28 14:26:28,696 ERROR 오류


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 기록은 지워지지 않는다</font></h3></td></tr></table>

같은 파일을 **한 번 더** 실행해, 기록이 지워지는지 이어 붙는지 확인하시오.

1. `!python my_log.py` 를 한 번 더 실행합니다.
2. `!cat my.log` 로 확인합니다.
3. 줄 수가 어떻게 바뀌었는지 봅니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 세 줄이 여섯 줄이 된다 |

**💡 힌트**

1. 같은 명령을 한 번 더 실행하면 됩니다.
2. `!cat` 으로 파일을 다시 봅니다.
3. 줄 수를 세어 봅니다.


In [44]:
!python my_log.py


In [45]:
!python my_log.py


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 처음 코딩하는 분은 건너뛰어도 됩니다. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-1 · 심각도를 올려 걸러내기</font></h3></td></tr></table>

`my_log.py` 의 설정에서 `level` 을 `logging.ERROR` 로 바꿔, **오류만** 기록에 남게 하시오.

1. 기록 파일 이름을 `only_error.log` 로 바꿉니다.
2. `logging.info`·`logging.warning`·`logging.error` 세 줄은 그대로 둡니다.
3. `!python` 으로 실행한 뒤 `!cat only_error.log` 로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `ERROR 오류` 한 줄만 남는다 |

**💡 힌트**

1. `level` 에 넣는 값만 바꿉니다.
2. `INFO` 보다 `ERROR` 가 더 심각합니다.
3. 기록 파일 이름을 바꿔야 앞 기록과 섞이지 않습니다.


In [ ]:
%%writefile only_error.py


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-2 · 기록 모양 바꾸기</font></h3></td></tr></table>

기록에서 심각도를 빼고 **시각과 내용만** 남게 하시오.

1. 설정의 `format` 에서 `%(levelname)s` 를 지웁니다.
2. 기록 파일 이름은 `short.log` 로 합니다.
3. `logging.warning` 으로 아무 문구나 하나 남기고 `!cat short.log` 로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `2026-09-28 10:00:00,000 내용` — `WARNING` 이 없다 |

**💡 힌트**

1. `format` 문자열에서 한 조각만 지웁니다.
2. 조각 사이의 빈칸도 같이 지웁니다.
3. `%(asctime)s` 와 `%(message)s` 만 남깁니다.


In [ ]:
%%writefile short_format.py


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 파서에 기록 붙이기</mark>

2교시에서 `print` 로 알리던 깨진 줄을 `logging.warning` 으로 바꿉니다.


In [ ]:
%%writefile parse_with_log.py
import logging

logging.basicConfig(
    filename="agent.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("파서 시작: sample_logs_broken.csv")

logs = []
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            logs.append(parts[3])
        except IndexError:
            logging.warning(f"깨진 줄 건너뜀: {line.strip()}")

logging.info(f"정상 로그 {len(logs)}건 처리 완료")
print(f"정상 로그 {len(logs)}건")


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-6 · 무엇이 보일까요</font></h3></td></tr></table>

위 셀로 `parse_with_log.py` 를 만들었습니다. 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
!python parse_with_log.py
```

막히면 바로 위 `4.2 파서에 기록 붙이기` 설명을 다시 봅니다.


In [47]:
!python parse_with_log.py


python3: can't open file '/content/drive/MyDrive/agent_core/parse_with_log.py': [Errno 2] No such file or directory


✅ `정상 로그 17건`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-7 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 그 실행이 남긴 `agent.log` 를 열어 봅니다.

```python
!cat agent.log
```

막히면 바로 위 `4.2 파서에 기록 붙이기` 설명을 다시 봅니다.


In [48]:
!cat agent.log


2026-09-28 14:24:21,895 INFO 기록을 시작합니다
2026-09-28 14:24:21,896 WARNING 주의할 일이 있었습니다


✅ `파서 시작 · 깨진 줄 건너뜀 5줄 · 정상 로그 17건 처리 완료`


앞서 남긴 기록이 지워지지 않고 **아래에 이어 붙었습니다.** `logging` 은 언제나 덧붙입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-8 · 시작과 끝을 기록에 남기기</font></h3></td></tr></table>

`sample_logs.csv`(깨끗한 파일)를 읽는 `.py` 파일을 만들고, 기록을 세 번 남기시오.

1. 새 셀 맨 첫 줄에 `%%writefile count_logs.py` 를 씁니다.
2. 설정 네 줄을 그대로 씁니다. 기록 파일 이름은 `agent.log` 입니다.
3. 파일을 읽기 전에 `logging.info` 로 `집계 시작` 을 남깁니다.
4. 파일을 한 줄씩 읽어 줄 수를 셉니다.
5. 다 읽은 뒤 `logging.info` 로 `집계 완료 17건` 을 남깁니다.
6. 마지막에 `!python count_logs.py` 와 `!cat agent.log` 를 각각 실행해 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `agent.log` 에 `집계 시작` 과 `집계 완료 17건` 이 이어 붙는다 |

**💡 힌트**

1. 설정은 문제 4-3에서 쓴 것과 같습니다.
2. 읽기 전과 읽은 뒤에 `logging.info` 를 한 번씩 둡니다.
3. 줄 수는 반복 안에서 1씩 더해 셉니다.


In [56]:
# 여기에 코드를 입력하세요
!python count_logs.py
!cat agent.log



17
2026-09-28 14:24:21,895 INFO 기록을 시작합니다
2026-09-28 14:24:21,896 WARNING 주의할 일이 있었습니다
2026-09-28 14:29:20,961 INFO 파서 시작: sample_logs_broken.csv
2026-09-28 14:29:20,962 WARNING 깨진 줄 건너뜀: 03:22,hacker
2026-09-28 14:29:20,962 WARNING 깨진 줄 건너뜀: 
2026-09-28 14:29:20,962 WARNING 깨진 줄 건너뜀: 서버 점검 안내: 오늘 밤 2시부터
2026-09-28 14:29:20,962 WARNING 깨진 줄 건너뜀: ??:??,unknown,LOGIN_FAIL
2026-09-28 14:29:20,963 WARNING 깨진 줄 건너뜀: 09:41,guest
2026-09-28 14:29:20,963 INFO 정상 로그 17건 처리 완료
2026-09-28 14:30:14,561 INFO 집계 시작
2026-09-28 14:30:14,563 INFO 집계 완료 17건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-9 · log_parser.py 조립하기</font></h3></td></tr></table>

오늘 배운 것을 전부 합쳐 **`log_parser.py`** 를 만드시오. 새 문법은 없습니다. 2·3교시의 조각을 잇는 시간입니다.

1. 새 셀 맨 첫 줄에 `%%writefile log_parser.py` 를 씁니다.
2. 설정 네 줄을 쓰고, `logging.info` 로 `파서 시작: sample_logs_broken.csv` 를 남깁니다.
3. `parse_line` 함수를 만듭니다. 줄을 나눠 `time`·`user`·`event`·`ip` 딕셔너리를 돌려줍니다.
4. 파일을 한 줄씩 읽어 `try` 안에서 `parse_line` 을 부르고, 결과를 리스트에 모읍니다.
5. `except IndexError:` 에서 `logging.warning` 으로 깨진 줄을 남깁니다.
6. 다 읽은 뒤 `logging.info` 로 `정상 로그 17건 처리 완료` 를 남깁니다.
7. 정상 로그에서 `LOGIN_FAIL` 인 계정을 세어, **3회 이상**인 계정을 화면에 출력합니다. 세는 도구는 지난주(9/23)에 배운 `Counter` 입니다.

| | |
|---|---|
| 🎯 화면 | `확인 필요: admin — 실패 3회` |
| 🎯 `agent.log` | 시작 · 경고 5건 · 완료 |

**💡 힌트**

1. 오늘 쓴 조각을 순서대로 이어 붙이는 것입니다.
2. `parse_line` 은 도전 ⭐2-5에서 만든 것과 같습니다.
3. 세는 도구는 `Counter` 이고 `from collections import Counter` 로 불러옵니다.


In [57]:
# 여기에 코드를 입력하세요
#@title 정답 4-9 { display-mode: "form" }
code = """import logging
from collections import Counter

logging.basicConfig(
    filename="agent.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)


def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]}


logging.info("파서 시작: sample_logs_broken.csv")

logs = []
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        try:
            logs.append(parse_line(line))
        except IndexError:
            logging.warning(f"깨진 줄 건너뜀: {line.strip()}")

logging.info(f"정상 로그 {len(logs)}건 처리 완료")

failed_users = []
for log in logs:
    if log["event"] == "LOGIN_FAIL":
        failed_users.append(log["user"])

counted = Counter(failed_users)
for user in counted:
    if counted[user] >= 3:
        print(f"확인 필요: {user} — 실패 {counted[user]}회")
"""

with open("log_parser.py", "w", encoding="utf-8") as f:
    f.write(code)

print("log_parser.py 를 만들었습니다. 새 셀에서 !python log_parser.py 를 실행하세요.")


log_parser.py 를 만들었습니다. 새 셀에서 !python log_parser.py 를 실행하세요.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-10 · 드라이브에 남았는지 확인하기</font></h3></td></tr></table>

맨 앞 준비 셀에서 `agent_core` 로 들어와 있으므로, 오늘 만든 파일은 이미 드라이브에 저장돼 있습니다. 확인하시오.

1. `!ls` 로 폴더 안을 봅니다.
2. `!python log_parser.py` 로 다시 돌아가는지 봅니다.
3. 드라이브 웹에서 `agent_core` 폴더를 열어 눈으로도 확인합니다.

| | |
|---|---|
| 🎯 확인 | `log_parser.py` 와 `sample_logs_broken.csv` 가 보이고, 실행하면 `정상 로그 17건` 이 나온다 |

**💡 힌트**

1. 맨 앞 준비 셀의 `%cd` 가 실행돼 있어야 합니다.
2. 출력에 `/content/drive/MyDrive/agent_core` 가 보이면 자리가 맞습니다.


In [55]:
!ls
!python log_parser.py


agent.log	    log_parser.py	  __pycache__
checks.py	    logs_reordered.csv	  report.json
count_logs.py	    logs_with_header.csv  sample_logs_broken.csv
day2_csv_report.py  my.log		  sample_logs.csv
day2_read_logs.py   my_log.py		  sample_logs_raw.csv
hello.py	    nolevel_demo.py	  summary.json
log_demo.py	    nolevel.log		  utils.py
확인 필요: admin — 실패 3회


맨 앞 준비 셀을 실행하지 않았다면 파일이 코랩 안에만 남습니다. 그 셀을 실행한 뒤 데이터 셀과 `%%writefile` 셀을 다시 실행하세요.


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>위 문제를 다 푼 사람만 풉니다. 새 문법은 나오지 않습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-3 · 깨진 이유를 기록에 나눠 남기기</font></h3></td></tr></table>

`log_parser.py` 의 `except` 를 두 개로 나누고, 각각 다른 경고 문구를 남기시오.

1. `except ValueError:` → `logging.warning(f"시각이 깨진 줄: …")`
2. `except IndexError:` → `logging.warning(f"칸이 모자란 줄: …")`
3. `parse_line` 안에서 시각의 시를 숫자로 바꿔야 `ValueError` 가 납니다.

| | |
|---|---|
| 🎯 `agent.log` | 시각이 깨진 줄 3건 · 칸이 모자란 줄 2건 |

**💡 힌트**

1. `parse_line` 안에 시를 숫자로 바꾸는 줄을 넣습니다.
2. 부르는 쪽 `except` 를 두 개로 나눕니다.
3. 문구만 다르고 뼈대는 같습니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-4 · 파일이 없을 때도 기록 남기기</font></h3></td></tr></table>

없는 파일을 열었을 때 프로그램이 멈추지 않고 `logging.error` 로 기록을 남기게 하시오.

1. 파일을 여는 `with` 전체를 `try` 로 감쌉니다.
2. `except FileNotFoundError:` 에서 `logging.error("로그 파일을 찾을 수 없음")` 을 남깁니다.
3. 파일 이름을 일부러 틀리게 적어 확인합니다.

| | |
|---|---|
| 🎯 `agent.log` | `ERROR 로그 파일을 찾을 수 없음` |

**💡 힌트**

1. 파일을 여는 부분 전체를 `try` 로 감쌉니다.
2. `logging.error` 는 `logging.warning` 과 쓰는 법이 같습니다.
3. 파일 이름을 일부러 틀리게 적어 확인합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-5 · 기록을 읽어 세기</font></h3></td></tr></table>

`agent.log` 를 **파일로 읽어** `WARNING` 이 몇 줄인지 세시오. 오늘 배운 파일 읽기와 조건문만 씁니다.

1. `agent.log` 를 한 줄씩 읽습니다.
2. 줄 안에 `WARNING` 이라는 글자가 들어 있으면 숫자를 1 늘립니다.
3. 마지막에 그 숫자를 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `경고 N건` (지금까지 실행한 횟수에 따라 달라집니다) |

**💡 힌트**

1. `agent.log` 도 그냥 글자 파일입니다. 파일 읽기와 같습니다.
2. 글자가 들어 있는지는 `in` 으로 확인합니다.
3. 숫자를 담을 이름을 반복 전에 만듭니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `logging.basicConfig(filename=…)` | 화면이 아니라 그 파일에 남긴다 |
| `level=logging.INFO` | `INFO` 부터 남긴다. 빼면 `WARNING` 부터 |
| `logging.info` | 정상 동작 기록 |
| `logging.warning` | 주의할 일 |
| `logging.error` | 심각한 오류 |
| `!cat 파일이름` | 파일 내용을 화면에 보여 준다 |

오늘의 산출물은 드라이브 `agent_core` 폴더의 **`log_parser.py`** 와 그것이 남긴 **`agent.log`** 입니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다. 정답 셀은 혼자서 실행됩니다.


In [ ]:
#@title 정답 2-3 { display-mode: "form" }
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts)


In [ ]:
#@title 정답 2-4 { display-mode: "form" }
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts[1])


In [ ]:
#@title 정답 2-5 { display-mode: "form" }
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(parts[0], parts[3])


In [ ]:
#@title 정답 ⭐2-1 { display-mode: "form" }
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")

print(f"{parts[1]} 의 접속 IP 는 {parts[3]} 입니다")


In [ ]:
#@title 정답 ⭐2-2 { display-mode: "form" }
line = "09:05,admin,LOGIN_FAIL,10.0.9.8"
parts = line.split(",")
time_parts = parts[0].split(":")

print(time_parts[0])
print(time_parts[1])


In [ ]:
#@title 정답 2-8 { display-mode: "form" }
line = "09:41,guest"
parts = line.split(",")

try:
    print(parts[3])
except IndexError:
    print("칸이 모자랍니다")

print("확인 끝")


In [ ]:
#@title 정답 2-9 { display-mode: "form" }
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            print(parts[1], parts[3])
        except IndexError:
            pass


In [ ]:
#@title 정답 2-10 { display-mode: "form" }
broken_count = 0

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            parts[3]
        except IndexError:
            broken_count = broken_count + 1

print(f"깨진 줄 {broken_count}건")


In [ ]:
#@title 정답 ⭐2-3 { display-mode: "form" }
broken_lines = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            parts[3]
        except IndexError:
            broken_lines.append(line.strip())

print(broken_lines)


In [ ]:
#@title 정답 ⭐2-4 { display-mode: "form" }
logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            logs.append({"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]})
        except IndexError:
            pass

print(f"정상 로그 {len(logs)}건")


In [ ]:
#@title 정답 ⭐2-5 { display-mode: "form" }
def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]}

logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        try:
            logs.append(parse_line(line))
        except IndexError:
            pass

print(f"정상 로그 {len(logs)}건")


In [ ]:
#@title 정답 3-3 { display-mode: "form" }
time = "21:30"

print(int(time.split(":")[0]))


In [ ]:
#@title 정답 3-4 { display-mode: "form" }
line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"
parts = line.split(",")

print(int(parts[0].split(":")[0]))


In [ ]:
#@title 정답 3-5 { display-mode: "form" }
line = "03:22,hacker,LOGIN_FAIL,10.0.9.9"
parts = line.split(",")
hour = int(parts[0].split(":")[0])

if hour >= 0 and hour <= 6:
    print("야간")


In [ ]:
#@title 정답 ⭐3-1 { display-mode: "form" }
times = ["09:01", "03:22", "21:30"]

for time in times:
    print(int(time.split(":")[0]))


In [ ]:
#@title 정답 ⭐3-2 { display-mode: "form" }
times = ["09:01", "03:22", "21:30"]

for time in times:
    hour = int(time.split(":")[0])
    if hour >= 0 and hour <= 6:
        print(hour, "야간")
    else:
        print(hour, "주간")


In [ ]:
#@title 정답 3-8 { display-mode: "form" }
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            hour = int(parts[0].split(":")[0])
            if hour >= 0 and hour <= 6:
                print(parts[1])
        except ValueError:
            pass
        except IndexError:
            pass


In [ ]:
#@title 정답 3-9 { display-mode: "form" }
bad_time = 0
bad_column = 0

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            int(parts[0].split(":")[0])
            parts[3]
        except ValueError:
            bad_time = bad_time + 1
        except IndexError:
            bad_column = bad_column + 1

print(f"시각이 깨진 줄 {bad_time}건")
print(f"칸이 모자란 줄 {bad_column}건")


In [ ]:
#@title 정답 3-10 { display-mode: "form" }
try:
    with open("sample_logs_2025.csv", encoding="utf-8") as f:
        print(f.readline())
except FileNotFoundError:
    print("파일이 없습니다")
finally:
    print("확인 끝")


In [ ]:
#@title 정답 ⭐3-3 { display-mode: "form" }
bad_time = []
bad_column = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            int(parts[0].split(":")[0])
            parts[3]
        except ValueError:
            bad_time.append(line.strip())
        except IndexError:
            bad_column.append(line.strip())

print(bad_time)
print(bad_column)


In [ ]:
#@title 정답 ⭐3-4 { display-mode: "form" }
def parse_line(line):
    parts = line.strip().split(",")
    return {
        "time": parts[0],
        "user": parts[1],
        "event": parts[2],
        "ip": parts[3],
        "hour": int(parts[0].split(":")[0]),
    }

logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        try:
            logs.append(parse_line(line))
        except ValueError:
            pass
        except IndexError:
            pass

print(f"정상 로그 {len(logs)}건")


In [ ]:
#@title 정답 ⭐3-5 { display-mode: "form" }
tried = 0
logs = []

with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        try:
            int(parts[0].split(":")[0])
            parts[3]
            logs.append(parts[1])
        except ValueError:
            pass
        except IndexError:
            pass
        finally:
            tried = tried + 1

print(f"시도한 줄 {tried}건")
print(f"정상 로그 {len(logs)}건")


In [ ]:
#@title 정답 4-3 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''%%writefile my_log.py
import logging

logging.basicConfig(
    filename="my.log",
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.warning("첫 기록")''')

In [ ]:
#@title 정답 4-4 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''%%writefile my_log.py
import logging

logging.basicConfig(
    filename="my.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("시작")
logging.warning("주의")
logging.error("오류")''')

In [ ]:
#@title 정답 4-5 { display-mode: "form" }
print("!python my_log.py 를 한 번 더 실행한 뒤 !cat my.log 로 봅니다.")
print("logging 은 언제나 파일 끝에 덧붙입니다. 세 줄이 여섯 줄이 됩니다.")


In [ ]:
#@title 정답 ⭐4-1 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''%%writefile only_error.py
import logging

logging.basicConfig(
    filename="only_error.log",
    level=logging.ERROR,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("시작")
logging.warning("주의")
logging.error("오류")''')

In [ ]:
#@title 정답 ⭐4-2 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''%%writefile short_format.py
import logging

logging.basicConfig(
    filename="short.log",
    format="%(asctime)s %(message)s",
    encoding="utf-8",
)

logging.warning("깨진 줄 건너뜀")''')

In [ ]:
#@title 정답 4-8 { display-mode: "form" }
code = """import logging

logging.basicConfig(
    filename="agent.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)

logging.info("집계 시작")

count = 0
with open("sample_logs.csv", encoding="utf-8") as f:
    for line in f:
        count = count + 1

logging.info(f"집계 완료 {count}건")
print(count)
"""

with open("count_logs.py", "w", encoding="utf-8") as f:
    f.write(code)

print("count_logs.py 를 만들었습니다. 아래 두 줄을 각각 새 셀에서 실행하세요.")
print("!python count_logs.py")
print("!cat agent.log")


In [ ]:
#@title 정답 4-9 { display-mode: "form" }
code = """import logging
from collections import Counter

logging.basicConfig(
    filename="agent.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    encoding="utf-8",
)


def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "event": parts[2], "ip": parts[3]}


logging.info("파서 시작: sample_logs_broken.csv")

logs = []
with open("sample_logs_broken.csv", encoding="utf-8") as f:
    for line in f:
        try:
            logs.append(parse_line(line))
        except IndexError:
            logging.warning(f"깨진 줄 건너뜀: {line.strip()}")

logging.info(f"정상 로그 {len(logs)}건 처리 완료")

failed_users = []
for log in logs:
    if log["event"] == "LOGIN_FAIL":
        failed_users.append(log["user"])

counted = Counter(failed_users)
for user in counted:
    if counted[user] >= 3:
        print(f"확인 필요: {user} — 실패 {counted[user]}회")
"""

with open("log_parser.py", "w", encoding="utf-8") as f:
    f.write(code)

print("log_parser.py 를 만들었습니다. 새 셀에서 !python log_parser.py 를 실행하세요.")


In [ ]:
#@title 정답 4-10 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''맨 앞 준비 셀을 실행했다면 이미 드라이브에 저장돼 있습니다.

확인
   !ls
   !python log_parser.py   → 정상 로그 17건

안 보이면
   준비 셀(%cd /content/drive/MyDrive/agent_core)을 실행하지 않은 것입니다.
   그 셀을 실행한 뒤 데이터 셀과 %%writefile log_parser.py 셀을 다시 실행합니다.''')

In [ ]:
#@title 정답 ⭐4-3 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''parse_line 안에 hour 를 추가하고, 부르는 쪽 except 를 둘로 나눕니다.

def parse_line(line):
    parts = line.strip().split(",")
    return {"time": parts[0], "user": parts[1], "event": parts[2],
            "ip": parts[3], "hour": int(parts[0].split(":")[0])}

        try:
            logs.append(parse_line(line))
        except ValueError:
            logging.warning(f"시각이 깨진 줄: {line.strip()}")
        except IndexError:
            logging.warning(f"칸이 모자란 줄: {line.strip()}")''')

In [ ]:
#@title 정답 ⭐4-4 { display-mode: "form" }
print("아래 내용을 **새 셀에 그대로** 붙여 넣고 실행하세요.\n")
print('''logging.info("파서 시작")

try:
    with open("sample_logs_2025.csv", encoding="utf-8") as f:
        for line in f:
            pass
except FileNotFoundError:
    logging.error("로그 파일을 찾을 수 없음")''')

In [ ]:
#@title 정답 ⭐4-5 { display-mode: "form" }
warning_count = 0

with open("agent.log", encoding="utf-8") as f:
    for line in f:
        if "WARNING" in line:
            warning_count = warning_count + 1

print(f"경고 {warning_count}건")
